# 🎤 YingMusic-SVC — Google Colab (Gradio)

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all` (หรือกด ▶ ทีละ Cell)
3. รอจน Cell 2 ขึ้น `Running on public URL: https://xxxx.gradio.live` → คลิกเปิดแล้วใช้งานได้เลย

ครั้งแรกใช้เวลาติดตั้ง + โหลดโมเดลประมาณ 5–8 นาที

In [ ]:
#@title 1. ติดตั้ง + โหลดโมเดล
REPO_URL = "https://github.com/DDME36/YingMusic-SVC"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"}
#@markdown ใส่ `HF_TOKEN` เฉพาะกรณีที่ Hugging Face ขอ login
HF_TOKEN = ""  #@param {type:"string"}

import os, sys, subprocess
if subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode != 0:
    raise SystemExit("❌ ไม่พบ GPU — ไปที่ Runtime → Change runtime type → เลือก T4 GPU แล้วรันใหม่")
if sys.version_info[:2] > (3, 12):
    print("⚠️ Python > 3.12: torch 2.4.0 อาจติดตั้งไม่ได้")

# --- clone ---
REPO_DIR = "/content/YingMusic-SVC"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    branch_flag = f"-b {BRANCH}" if BRANCH else ""
    !git clone -q --depth 1 $branch_flag $REPO_URL $REPO_DIR
%cd $REPO_DIR

# --- dependencies ---
# SoX จำเป็นสำหรับ echo/reverb ตอนมิกซ์กลับ; ล็อก torch 2.4.0 เพราะ torchaudio >= 2.9 ตัด sox_effects ออก
print("กำลังติดตั้ง dependencies ...")
!apt-get -qq update > /dev/null && apt-get -qq install -y sox libsox-dev libsox-fmt-all ffmpeg > /dev/null
TORCH_PINS = "torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0"
!pip install -q $TORCH_PINS --index-url https://download.pytorch.org/whl/cu121
!pip install -q $TORCH_PINS --extra-index-url https://download.pytorch.org/whl/cu121 \
    "numpy==1.26.4" "scipy==1.13.1" "librosa==0.10.2" "transformers==4.46.3" \
    "huggingface-hub>=0.28.1,<1.0" "munch==4.0.0" "einops==0.8.0" "descript-audio-codec==1.0.0" \
    "gradio==5.23.0" "pydantic==2.10.6" "pydub==0.25.1" "soundfile==0.12.1" pyyaml \
    omegaconf ml_collections loralib wandb "beartype==0.14.1" "rotary_embedding_torch==0.3.5" matplotlib

# --- models ---
print("กำลังโหลดโมเดล ...")
from huggingface_hub import hf_hub_download, snapshot_download
token = HF_TOKEN or None
SVC_CKPT = os.path.abspath(hf_hub_download("GiantAILab/YingMusic-SVC", "YingMusic-SVC-full.pt",
                                           local_dir="checkpoints", token=token))
hf_hub_download("GiantAILab/YingMusic-SVC", "bs_roformer.ckpt",
                local_dir="accom_separation/ckpt/bs_roformer", token=token)
# โมเดลเสริม: โหลดล่วงหน้าเข้า cache path เดียวกับที่ my_inference.py / hf_utils.py ใช้
hf_hub_download("lj1995/VoiceConversionWebUI", "rmvpe.pt", cache_dir="./checkpoints")
hf_hub_download("funasr/campplus", "campplus_cn_common.bin", cache_dir="./checkpoints")
snapshot_download("nvidia/bigvgan_v2_44khz_128band_512x", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["config.json", "bigvgan_generator.pt"])
snapshot_download("openai/whisper-small", cache_dir="./checkpoints/hf_cache",
                  allow_patterns=["*.json", "*.safetensors", "*.txt"])

os.environ["YINGMUSIC_SVC_CKPT"] = SVC_CKPT
!python -c "import torch, torchaudio; from torchaudio import sox_effects; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
print("✅ พร้อมแล้ว — รัน Cell 2 เพื่อเปิด Gradio")

### วิธีใช้ใน Gradio
- **⚡ One-Click Separation + SVC** — อัปโหลดเพลงเต็ม (มีดนตรี) + เสียงเป้าหมาย → กด *Process All* ได้ทั้งเสียงร้องที่แปลงแล้วและเพลงที่มิกซ์กลับ
- **🎤 Standalone SVC** — ถ้ามีไฟล์เสียงร้องล้วนอยู่แล้ว
- **🎚️ Accompaniment Separation** — แยกเสียงร้อง/ดนตรีอย่างเดียว

เสียงเป้าหมายควรเป็นเสียงร้อง/พูดที่สะอาด ไม่มีดนตรี ยาว 10–25 วินาที (หรือกด *Use Sample Timbre*) · กด Convert ครั้งแรกจะช้า (~1 นาที, โหลดโมเดล) ครั้งต่อไปเร็วขึ้น · Cell 2 จะรันค้างไว้ตลอด ถ้าปิดต้องรันใหม่

In [ ]:
#@title 2. เปิด Gradio
%cd /content/YingMusic-SVC
!PYTHONUNBUFFERED=1 python -u -c "import gradio_app as g; g.build_ui().queue().launch(share=True, server_name='0.0.0.0', server_port=7860)"